<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c03-header.png" alt="Nextia Learning · Essential Mathematics and Statistics for AI" width="100%">

# Solution: Optimization behaviour

**[Optimization behaviour](https://learning.nextia-ai.com/courses/math/m04/optimization-behaviour/)** · Essential Mathematics and Statistics for AI · Module 4, lesson 3 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**Optional: check the numbers in Python.** This notebook is optional. The lesson works without code: a calculator is enough. Use the notebook to check the lesson's numbers and to redo its worked examples and tasks in Python.

**You will** run gradient descent on the ticket model with many learning rates, see a run converge, zig-zag and diverge to `inf` and `nan`, find a local minimum, run stochastic gradient descent with the explorer's batches, and do the module practice.

| | |
|---|---|
| **Time** | About 30 minutes, after you read the lesson |
| **Needs** | Nothing to install and no account: only Python's standard library. It runs in Colab, Kaggle or any Jupyter. |
| **You should know** | The loss of the ticket model, the gradient 15w − 94 and one step of the update rule, from [Loss and gradients: The update rule](https://learning.nextia-ai.com/courses/math/m04/loss-and-gradients/#the-update-rule-step-downhill) and [Worked example: two steps by hand](https://learning.nextia-ai.com/courses/math/m04/loss-and-gradients/#worked-example-two-steps-by-hand). |
| **Tested** | 2026-10-07, Python 3.14.6 (standard library only), in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

The [lesson page](https://learning.nextia-ai.com/courses/math/m04/optimization-behaviour/) has the full explanations, the charts, the explorers and the knowledge checks that count toward your certificate. Read a section there, then check its numbers here.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, replace each `...` with your calculation, run the cell, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C03/M04-L03-optimization-behaviour/optimization-behaviour-solution.ipynb).

## Setup: the helpers

The next cell defines three small functions that the notebook uses. You do not need to read them, but you can.

- `expect(what, yours, expected)` compares your number with the lesson's number. It says "Check passed." or if yours is too high or too low. It does not show the answer.
- `table(headers, rows)` prints a small table with aligned columns.
- `bars(labels, counts)` prints a text bar chart: one row of `█` for each label.

Run the cell. You should see `Ready.`

In [ ]:
# Helpers: check an answer without showing it, print a table, draw text bars.
import math, random, statistics

def expect(what, yours, expected, tol=0.01):
    """Compare your value with the expected one, to within `tol` for numbers."""
    if yours is ...:
        print(f"Not yet: replace ... with your calculation for {what}, then run the cell again.")
        return
    if isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        if abs(yours - expected) <= tol:
            print(f"Check passed: {what} is {yours:g}.")
        else:
            side = "too high" if yours > expected else "too low"
            print(f"Not yet: your {what}, {yours:g}, is {side}. Look at the step before it again.")
    elif yours == expected:
        print(f"Check passed: {what} is {yours}.")
    else:
        print(f"Not yet: your {what} is {yours!r}. That is not the expected answer.")

def table(headers, rows, digits=2):
    """Print rows as a table. Floats are rounded to `digits` decimals."""
    cell = lambda v: f"{v:,.{digits}f}" if isinstance(v, float) else str(v)
    text = [[cell(v) for v in row] for row in rows]
    widths = [max(len(str(h)), *(len(r[i]) for r in text)) for i, h in enumerate(headers)]
    print("  ".join(str(h).rjust(w) for h, w in zip(headers, widths)))
    for r in text:
        print("  ".join(v.rjust(w) for v, w in zip(r, widths)))

def bars(labels, counts, width=40):
    """Print a text bar chart: the longest bar is `width` characters."""
    top = max(counts) or 1
    size = max(len(str(label)) for label in labels)
    for label, n in zip(labels, counts):
        print(f"{str(label).rjust(size)} | {'█' * round(n / top * width)} {n}")

print("Ready.")

## Setup: the ticket model

The next cell defines the four tickets, the loss and the gradient from the last lesson, and two functions:

- `descend(rate, start, steps)` makes steps of gradient descent and returns the list of `w` values;
- `trace(w)` draws one line for a value of `w`: `·` is the axis from 0 to 12, `|` is the best `w` (6.27), and `●` is `w`.

Run the cell. You should see `Ready: loss at w = 2 is 137.75, gradient -64`.

In [ ]:
TICKETS = {"T-3101": (1, 7), "T-3102": (2, 13), "T-3103": (3, 17), "T-3104": (4, 26)}
BEST_W = 188 / 30            # about 6.27
LOWEST = 1183 / 4 - 188 ** 2 / 120   # about 1.217

def loss(w, batch=TICKETS):
    """Mean squared error on some tickets (all four by default)."""
    return sum((w * x - y) * (w * x - y) for x, y in (TICKETS[t] for t in batch)) / len(batch)

def gradient(w, batch=TICKETS):
    """Slope of that loss: the mean of 2 × (prediction − actual) × length. On all four: 15w − 94."""
    return sum(2 * (w * x - y) * x for x, y in (TICKETS[t] for t in batch)) / len(batch)

def descend(rate, start=2, steps=20):
    """Gradient descent on all four tickets. Returns w at the start and after each step."""
    ws = [start]
    for _ in range(steps):
        ws.append(ws[-1] - rate * gradient(ws[-1]))
    return ws

def trace(w, low=0, high=12, width=48):
    """One line: · for the w axis, | at the best w, ● at w (an arrow if w is off the axis)."""
    line = ["·"] * (width + 1)
    line[round((BEST_W - low) / (high - low) * width)] = "|"
    if w < low:
        line[0] = "◀"
    elif w > high:
        line[-1] = "▶"
    else:
        line[round((w - low) / (high - low) * width)] = "●"
    return "".join(line)

print(f"Ready: loss at w = 2 is {loss(2)}, gradient {gradient(2):g}")

## Setup: the explorer's ticket order

The lesson's batch explorer shuffles the four tickets at the start of each epoch, always in the same way. The next cell defines `orders`: the order of the tickets in each of the first 40 epochs, exactly as in the explorer. Section 5 uses it.

Run the cell. You should see `Ready: 40 epochs, epoch 1 is ['T-3104', 'T-3101', 'T-3103', 'T-3102']`.

In [ ]:
# The data, written in full (you do not need to read it).
orders = [
    ["T-3104", "T-3101", "T-3103", "T-3102"], ["T-3104", "T-3101", "T-3103", "T-3102"],
    ["T-3104", "T-3103", "T-3101", "T-3102"], ["T-3104", "T-3101", "T-3103", "T-3102"],
    ["T-3103", "T-3102", "T-3101", "T-3104"], ["T-3103", "T-3102", "T-3104", "T-3101"],
    ["T-3103", "T-3101", "T-3102", "T-3104"], ["T-3102", "T-3104", "T-3101", "T-3103"],
    ["T-3101", "T-3104", "T-3102", "T-3103"], ["T-3102", "T-3103", "T-3101", "T-3104"],
    ["T-3102", "T-3101", "T-3103", "T-3104"], ["T-3104", "T-3103", "T-3102", "T-3101"],
    ["T-3102", "T-3104", "T-3101", "T-3103"], ["T-3102", "T-3103", "T-3101", "T-3104"],
    ["T-3103", "T-3101", "T-3102", "T-3104"], ["T-3103", "T-3104", "T-3101", "T-3102"],
    ["T-3104", "T-3102", "T-3101", "T-3103"], ["T-3104", "T-3103", "T-3101", "T-3102"],
    ["T-3101", "T-3103", "T-3104", "T-3102"], ["T-3103", "T-3101", "T-3102", "T-3104"],
    ["T-3102", "T-3101", "T-3103", "T-3104"], ["T-3102", "T-3103", "T-3104", "T-3101"],
    ["T-3103", "T-3101", "T-3102", "T-3104"], ["T-3103", "T-3101", "T-3104", "T-3102"],
    ["T-3104", "T-3101", "T-3103", "T-3102"], ["T-3104", "T-3101", "T-3103", "T-3102"],
    ["T-3104", "T-3103", "T-3102", "T-3101"], ["T-3104", "T-3103", "T-3101", "T-3102"],
    ["T-3104", "T-3103", "T-3102", "T-3101"], ["T-3102", "T-3103", "T-3101", "T-3104"],
    ["T-3101", "T-3103", "T-3102", "T-3104"], ["T-3103", "T-3101", "T-3102", "T-3104"],
    ["T-3104", "T-3103", "T-3102", "T-3101"], ["T-3102", "T-3103", "T-3104", "T-3101"],
    ["T-3102", "T-3101", "T-3103", "T-3104"], ["T-3102", "T-3103", "T-3104", "T-3101"],
    ["T-3103", "T-3102", "T-3104", "T-3101"], ["T-3103", "T-3104", "T-3102", "T-3101"],
    ["T-3104", "T-3102", "T-3101", "T-3103"], ["T-3102", "T-3104", "T-3103", "T-3101"],
]
print(f"Ready: {len(orders)} epochs, epoch 1 is {orders[0]}")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Learning rate: too small, good, too large

The gradient is 15 × (w − 6.27), so it is proportional to the distance from the bottom. Each step therefore multiplies that distance by the same number: **1 − 15 × learning rate**.

> **Predict.** For which learning rate is this factor 0? For which is it −1? Then run the cell.

In [ ]:
rows = []
for rate in [0.01, 0.05, 1 / 15, 0.1, 0.13, 2 / 15, 0.14]:
    factor = 1 - 15 * rate
    if factor > 0:
        what = "smooth"
    elif abs(factor) < 1e-9:
        what = "lands on the bottom"
    elif factor > -1:
        what = "zig-zag that closes in"
    elif abs(factor + 1) < 1e-9:
        what = "bounces forever"
    else:
        what = "diverges"
    rows.append([round(rate, 4), round(factor, 4), what])
table(["Learning rate", "1 − 15 × rate", "Behaviour"], rows, digits=4)

> **Check.** You should see the factor 0 at 0.0667 (= 1/15) and −1 at 0.1333 (= 2/15). Below 0.067 the steps are smooth. Between 0.067 and 0.133 they zig-zag to the bottom. Above 0.133 the run diverges. These limits belong to this curve only.

Now watch four runs from `w = 2`, as in the lesson's first explorer. Each line of the trace is one step.

> **Predict.** Which run reaches the bottom fastest: 0.01, 0.06 or 0.12? What happens at 0.14? Then run the cell.

In [ ]:
for rate in [0.01, 0.06, 0.12, 0.14]:
    ws = descend(rate, steps=3)
    print(f"learning rate {rate}")
    for step, w in enumerate(ws):
        print(f"  step {step}  {trace(w)}  w = {w:6.2f}   loss {loss(w):7.1f}")
    print()

> **Check.** You should see:
>
> - 0.01: `w` goes 2, 2.64, 3.18, 3.65: slow, small steps.
> - 0.06: `w` goes 2, 5.84, 6.22, 6.26: smooth and fast.
> - 0.12: `w` goes 2, 9.68, 3.54, 8.45: a zig-zag around the `|`.
> - 0.14: `w` goes 2, 10.96, 1.10, 11.95, and the loss grows: 137.8, 166.4, 201.1, 243.1.

> **Your turn: change one thing.** In the cell above, change the list to `[0.1, 0.13, 0.16]` and `steps=3` to `steps=8`. At 0.16, `w` leaves the axis after a few steps: the trace shows `◀` or `▶`.

## 2. What a training loss curve looks like

A training log shows only the loss after each step. The next cell prints the loss after steps 1, 4, 10 and 20 for three learning rates, as in the lesson's chart.

> **Predict.** The runs with 0.01 and 0.13 both fall slowly. Why? Then run the cell.

In [ ]:
rows = []
for rate in [0.01, 0.05, 0.13]:
    ws = descend(rate, steps=20)
    rows.append([rate] + [round(loss(ws[i]), 2) for i in [1, 4, 10, 20]])
table(["Learning rate", "After 1", "After 4", "After 10", "After 20"], rows)

> **Check.** You should see 99.86, 38.42, 6.51 and 1.42 for 0.01; 9.75 after one step and 1.22 from step 4 on for 0.05; and 124.44, 91.8, 50.16 and 18.76 for 0.13. At 0.01, the steps are too short. At 0.13, they are too long: `w` jumps from side to side. The loss curve alone does not tell you which one.

When the learning rate is too large, the loss grows. Run the cell. It shows the loss for 12 steps as text bars, for 0.12 and 0.14.

In [ ]:
for rate in [0.12, 0.14]:
    ws = descend(rate, steps=12)
    print(f"learning rate {rate}")
    bars([f"step {i:2}" for i in range(13)], [round(loss(w), 1) for w in ws])
    print()

> **Check.** At 0.12, the bars get shorter: 137.8, then 88.6, 57.1, 37.0, and 1.9 after 12 steps. At 0.14, they get longer at every step: 166.4, 201.1, 243.1, and 1,346.0 after 12 steps. A small change in the learning rate turns a run that converges into a run that diverges.

### From a large loss to `inf` and `nan`

A diverging loss grows by the same factor at every step: about 1.21 at 0.14. Most training stores numbers in 32 bits, and the largest 32-bit number is about 3.4 × 10³⁸.

> **Predict.** At learning rate 0.16, after how many steps is the loss larger than 3.4 × 10³⁸? Then run the cell.

In [ ]:
w, step = 2, 0
while loss(w) <= 3.4e38:
    w = w - 0.16 * gradient(w)
    step += 1
print("the loss passes 3.4e38 after", step, "steps")

> **Check.** You should see `125` steps. In a 32-bit training run, the loss is now stored as `inf` (infinity).

Python stores numbers in 64 bits, up to about 1.8 × 10³⁰⁸. So the run goes much further before Python gives `inf`. Run the cell. You should see `inf` after 1046 steps. Then one more calculation with `inf` gives `nan`, "not a number": the `loss: nan` of a training log.

In [ ]:
w, step = 2, 0
while loss(w) != math.inf:
    w = w - 0.16 * gradient(w)
    step += 1
print("loss after", step, "steps:", loss(w))
print("inf - inf =", math.inf - math.inf)

## 3. Convergence

A run **converges** when its steps become so small that `w` and the loss stop changing. The next cell counts the steps until the loss, rounded to 3 decimals, shows the lowest value, 1.217.

> **Predict.** Which learning rate needs the fewest steps? Then run the cell.

In [ ]:
def steps_to_converge(rate, start=2, limit=500):
    ws = descend(rate, start, limit)
    for step, w in enumerate(ws):
        if round(loss(w), 3) == round(LOWEST, 3):
            return step
    return None

table(["Learning rate", "Steps to 1.217"], [[rate, steps_to_converge(rate)] for rate in [0.01, 0.02, 0.05, 0.06, 0.1, 0.12]])

> **Check.** You should see 37, 17, 5, 3, 9 and 27 steps. The good rates are near 1/15 = 0.067: 0.05 needs 5 steps, and 0.01 needs more than 30. Both a too-small and a too-large rate are slow.

## 4. Local minima: the start matters

The lesson's second explorer has a made-up loss curve with two valleys: loss = 0.08w⁴ − 0.9w² + 0.35w + 3.5. A **local minimum** is lower than everything near it, but not the lowest anywhere. The gradient there is 0, so gradient descent stops.

> **Predict.** The ball starts at `w = 1`, right of the hump, with learning rate 0.1. Does it find the deeper valley on the left? Then run the cell.

In [ ]:
def bumpy(w):
    return 0.08 * w ** 4 - 0.9 * w ** 2 + 0.35 * w + 3.5

def bumpy_slope(w):
    return 0.32 * w ** 3 - 1.8 * w + 0.35

def roll(start, rate=0.1, steps=200):
    w = start
    for _ in range(steps):
        w = w - rate * bumpy_slope(w)
    return w

for start in [1, -0.5]:
    end = roll(start)
    print(f"start {start:5}: stops at w = {end:.3f}, loss {bumpy(end):.3f}, slope {round(bumpy_slope(end), 4) + 0.0:.4f}")

> **Check.** From `w = 1`, the ball stops at `w = 2.268` with a loss of 1.781: the shallow valley, a local minimum. From `w = -0.5`, it stops at `w = -2.464` with a loss of 0.122: the deeper valley. In both places, the slope is 0.

> **Your turn.** Find the start where the result changes. Try the starts 0, 0.1, 0.2 and 0.3 in the cell above. You should see that 0.1 still rolls left and 0.2 rolls right: the top of the hump is between them.

## 5. Stochastic updates: why real loss curves are noisy

Real training computes the gradient on a **mini-batch**, a small random group of examples. One pass through all examples is an **epoch**. This is **stochastic gradient descent** (SGD).

Run the cell. It prints the gradient at `w = 6` on each one-ticket batch and on two batches of two tickets. You should see −2, −4, 6 and −16, then −3 and −5. The true gradient on all four is −4.

In [ ]:
rows = [[t, gradient(6, [t])] for t in TICKETS]
rows += [["T-3101 and T-3102", gradient(6, ["T-3101", "T-3102"])],
         ["T-3103 and T-3104", gradient(6, ["T-3103", "T-3104"])],
         ["All four", gradient(6)]]
table(["Batch", "Gradient at w = 6"], rows, digits=1)

A one-ticket batch can give a gradient four times too large (T-3104), or one that points the wrong way (T-3103).

Now run SGD as in the lesson's batch explorer: from `w = 5.5`, learning rate 0.02, with the explorer's ticket order `orders`. The next cell defines `sgd(batch_size, steps)` and prints the first 6 steps with one ticket per batch.

> **Predict.** At step 3, the batch is T-3103. Does that step go the right way? Then run the cell.

In [ ]:
def sgd(batch_size, steps=40, start=5.5, rate=0.02):
    """Stochastic gradient descent with the explorer's order. Returns one row per step."""
    w = start
    rows = []
    per_epoch = len(TICKETS) // batch_size
    for n in range(steps):
        epoch, k = divmod(n, per_epoch)
        batch = orders[epoch][k * batch_size:(k + 1) * batch_size]
        batch_grad, true_grad, batch_loss = gradient(w, batch), gradient(w), loss(w, batch)
        w = w - rate * batch_grad
        rows.append([n + 1, epoch + 1, " + ".join(batch), round(batch_loss, 3), round(batch_grad, 3), round(true_grad, 3), round(w, 3)])
    return rows

table(["Step", "Epoch", "Batch", "Batch loss", "Batch gradient", "True gradient", "w after"], sgd(1, 6), digits=3)

> **Check.** Step 1 uses T-3104: its gradient is −32, but the true gradient is −11.5. At step 3, T-3103 gives +9.139 while the true gradient is −1.384: the step goes the wrong way, and `w` falls from 6.174 to 5.992.

The next cell runs 40 steps with batches of 1, 2 and 4 tickets. It prints the range of `w` and of the batch loss over steps 21 to 40, and a trace of `w` for the last 8 steps, zoomed in to 5.9 to 6.6.

> **Predict.** With one ticket per batch, does `w` settle at 6.27? Then run the cell.

In [ ]:
for size in [1, 2, 4]:
    late = sgd(size, 40)[20:]
    ws = [r[6] for r in late]
    losses = [r[3] for r in late]
    print(f"batch of {size}: w from {min(ws):.3f} to {max(ws):.3f}; batch loss from {min(losses):.3f} to {max(losses):.3f}")
    for r in late[-8:]:
        print(f"   step {r[0]}  {trace(r[6], low=5.9, high=6.6, width=35)}  w = {r[6]:.3f}")
    print()

> **Check.** You should see:
>
> - batch of 1: `w` keeps moving between 6.010 and 6.492, and the batch loss jumps between 0.039 and 6.123;
> - batch of 2: `w` stays between 6.164 and 6.375;
> - batch of 4 (all tickets): every step is exact, `w` stays at 6.266 to 6.267 and the batch loss is 1.217.

The batch loss jumps because each ticket has its own error, not because the model gets worse. Read the **trend** over many steps, not one step.

## 6. Guided practice: diagnose three runs

Each list is the loss every 100 steps, from the lesson's three logs. The next cell prints the change from each report to the next. Run it.

In [ ]:
runs = {
    "A": [0.693, 0.691, 0.689, 0.687, 0.685, 0.683],
    "B": [0.693, 0.512, 0.388, 0.341, 0.330, 0.327],
    "C": [0.693, 0.590, 0.731, 0.552, 0.802, 0.517],
}
for name, values in runs.items():
    changes = [round(b - a, 3) for a, b in zip(values, values[1:])]
    print(f"Run {name}: changes {changes}")

> **Your turn.** For each run, write the most likely cause: `"too small"`, `"good"` or `"too large"` (the learning rate). Then run the check cell. The lesson's example answer gives one action for each run.

In [ ]:
diagnosis_a = "too small"   # it falls by only 0.002 per report, a straight, slow line
diagnosis_b = "good"        # it falls fast, then flattens near 0.33
diagnosis_c = "too large"   # it jumps, and the high values get higher

In [ ]:
expect("diagnosis of run A", diagnosis_a, "too small")
expect("diagnosis of run B", diagnosis_b, "good")
expect("diagnosis of run C", diagnosis_c, "too large")

## 7. Module practice: trace gradient descent

### Worked example: learning rate 0.1

The next cell prints Priya's table: from `w = 2` with learning rate 0.1, four steps. Run it. You should see `w` go 2, 8.4, 5.2, 6.8, 6.0, and the loss 35.35, 9.75, 3.35, 1.75.

In [ ]:
def trace_table(rate, steps, start=2):
    w = start
    rows = []
    for step in range(1, steps + 1):
        g = gradient(w)
        move = -rate * g
        rows.append([step, round(w, 4), round(g, 4), round(move, 4), round(w + move, 4), round(loss(w + move), 2)])
        w = w + move
        print(f"step {step}  {trace(w)}  w = {w:.4g}")
    print()
    table(["Step", "w before", "Gradient", "Move", "w after", "Loss after"], rows, digits=4)

trace_table(0.1, 4)

> **Check.** The sign of the gradient changes at every step, so `w` jumps from one side of 6.27 to the other. Each jump is half as long as the one before (6.4, 3.2, 1.6, 0.8): the factor is 1 − 15 × 0.1 = −0.5. The loss still falls at every step.

### Your task: two other learning rates

> **Your turn.** Make your tables in `c03-notes.md` first, with a calculator. Then replace each `...` below:
>
> 1. Learning rate 0.02, three steps from `w = 2`: `w` after the third step, and the loss there.
> 2. Learning rate 0.15, four steps from `w = 2`: `w` after the fourth step, and the loss there.
> 3. For each run, your verdict: `"too small"`, `"good"` or `"too large"`.
>
> Use the update rule in the loops, and `loss(...)`. Do not use `trace_table` for this task. Then run the check cell.

In [ ]:
w = 2
for step in range(3):
    w = w - 0.02 * gradient(w)
w_rate_002 = w
loss_rate_002 = loss(w)
verdict_002 = "too small"

w = 2
for step in range(4):
    w = w - 0.15 * gradient(w)
w_rate_015 = w
loss_rate_015 = loss(w)
verdict_015 = "too large"

Run the check cell.

In [ ]:
expect("w after 3 steps at 0.02", w_rate_002, 4.8032, tol=0.0001)
expect("the loss after 3 steps at 0.02", loss_rate_002, 17.28, tol=0.01)
expect("the verdict for 0.02", verdict_002, "too small")
expect("w after 4 steps at 0.15", w_rate_015, -4.15, tol=0.0001)
expect("the loss after 4 steps at 0.15", loss_rate_015, 815.02, tol=0.01)
expect("the verdict for 0.15", verdict_015, "too large")

To see the evidence, run `trace_table(0.02, 3)` and `trace_table(0.15, 4)` in a new cell. At 0.02, each step keeps 70% of the distance. At 0.15, each jump is 25% longer than the one before, and the loss grows at every step. The model answer with both tables is on the lesson page.

## Check your understanding and recap

Answer the **knowledge checks** and the **module check** on the [lesson page](https://learning.nextia-ai.com/courses/math/m04/optimization-behaviour/#module-check). They count toward your certificate when you are signed in and enrolled. The notebook does not have them.

In this notebook you ran gradient descent on the ticket model. Each step multiplies the distance to the bottom by 1 − 15 × learning rate: below 0.067 the run is smooth, up to 0.133 it zig-zags to the bottom, and above 0.133 it diverges, to `inf` and then `nan`. On a curve with two valleys, the start decides where the ball stops. With mini-batches, each step uses a noisy gradient, so `w` and the batch loss keep moving: read the trend.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Samples and populations](https://learning.nextia-ai.com/courses/math/m05/samples-and-populations/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/math/m04/optimization-behaviour/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The Larkfield tickets are fictional and have no real people. The training logs and the curve with two valleys are made up for teaching.